# Week 2 — Classical NLP & Embedding-Based Models
## Quora Duplicate Question Detection

**Goal:** Compare classical ML approaches (TF-IDF n-grams) with embedding-based approaches (Word2Vec)  
**Models compared:** Logistic Regression vs Linear SVM  
**Dataset:** `cleaned_quora.csv` — output from Week 1 EDA pipeline

## 1. Imports & Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import re
from collections import Counter

# NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.models import Word2Vec

# Feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Downloads
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print('All imports successful.')

All imports successful.


## 2. Load Cleaned Dataset

We load `cleaned_quora.csv` — the preprocessed output from Week 1 EDA.  
It already contains `clean_q1` and `clean_q2` columns (lowercased, special chars removed).

In [ ]:
import os
# Prefer parquet if available, fall back to CSV
if os.path.exists('cleaned_quora.parquet'):
    df = pd.read_parquet('cleaned_quora.parquet')
elif os.path.exists('cleaned_quora.csv'):
    df = pd.read_csv('cleaned_quora.csv')
else:
    raise FileNotFoundError('cleaned_quora.parquet or cleaned_quora.csv not found')

# Ensure clean text columns have no nulls
df['clean_q1'] = df['clean_q1'].fillna('').astype(str)
df['clean_q2'] = df['clean_q2'].fillna('').astype(str)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_quora.csv'

---
## 3. Approach 1 — TF-IDF with N-grams

**TF-IDF (Term Frequency-Inverse Document Frequency)** converts text into numerical vectors.  
- **TF** — how often a word appears in a document  
- **IDF** — penalises words that appear in many documents (common words carry less signal)  

**N-grams** capture multi-word phrases:  
- `ngram_range=(1,1)` → unigrams only (e.g., `"invest"`, `"market"`)  
- `ngram_range=(1,2)` → unigrams + bigrams (e.g., `"share market"`, `"step guide"`)  

We use `(1,2)` to capture two-word phrases that are more meaningful than individual words.

### 3.1 Build TF-IDF Feature Matrix

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,   # keep top 5000 terms by TF-IDF score
    stop_words='english',
    ngram_range=(1, 2)   # unigrams and bigrams
)

# Fit on q1 only — transform q2 separately to prevent data leakage
q1_tfidf = tfidf.fit_transform(df['clean_q1'])
q2_tfidf = tfidf.transform(df['clean_q2'])

# Stack both question vectors horizontally → 10,000-dim feature vector per row
X_tfidf = sp.hstack([q1_tfidf, q2_tfidf])
y = df['label']

print(f'TF-IDF Feature matrix shape: {X_tfidf.shape}')
print(f'Label distribution:\n{y.value_counts()}')

### 3.2 Sample N-gram Features

Let's inspect what unigrams and bigrams the vectorizer extracted.

In [ ]:
feature_names = tfidf.get_feature_names_out()

unigrams = [f for f in feature_names if ' ' not in f]
bigrams  = [f for f in feature_names if ' ' in f]

print(f'Total features: {len(feature_names)}')
print(f'Unigrams: {len(unigrams)}  |  Bigrams: {len(bigrams)}')
print(f'\nSample unigrams: {unigrams[:15]}')
print(f'Sample bigrams : {bigrams[:15]}')

### 3.3 Train-Test Split

We use an **80/20 stratified split** so both train and test sets maintain the same
63% / 37% class ratio as the full dataset. `random_state=42` ensures reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train size : {X_train.shape[0]:,}')
print(f'Test size  : {X_test.shape[0]:,}')
print(f'Train label distribution:\n{y_train.value_counts()}')

---
## 4. Model 1 — Logistic Regression

**Why Logistic Regression?**  
- Fast and interpretable baseline for binary classification  
- Works well on sparse, high-dimensional TF-IDF matrices  
- Outputs calibrated probabilities (useful for threshold tuning)  
- `max_iter=1000` ensures convergence on a large dataset (400K+ rows)

### 4.1 Train

In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
print('Logistic Regression training complete.')

### 4.2 Evaluate

In [ ]:
lr_pred = lr_model.predict(X_test)

print('=== Logistic Regression Results ===')
print(f'Accuracy : {accuracy_score(y_test, lr_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, lr_pred):.4f}')
print()
print(classification_report(y_test, lr_pred, target_names=['Non-Duplicate', 'Duplicate']))

### 4.3 Confusion Matrix

In [ ]:
lr_cm = confusion_matrix(y_test, lr_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    lr_cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Non-Dup', 'Duplicate'],
    yticklabels=['Non-Dup', 'Duplicate']
)
plt.title('Logistic Regression — Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
## 5. Model 2 — Linear SVM

**Why Linear SVM?**  
- Maximises the margin between classes — often more robust than LR on text data  
- `LinearSVC` is much faster than kernel SVM on large sparse matrices  
- Tends to outperform LR on high-dimensional TF-IDF features due to its maximum-margin objective  
- A strong classical NLP baseline alongside Logistic Regression

### 5.1 Train

In [ ]:
svm_model = LinearSVC(max_iter=1000)
svm_model.fit(X_train, y_train)
print('Linear SVM training complete.')

### 5.2 Evaluate

In [ ]:
svm_pred = svm_model.predict(X_test)

print('=== Linear SVM Results ===')
print(f'Accuracy : {accuracy_score(y_test, svm_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, svm_pred):.4f}')
print()
print(classification_report(y_test, svm_pred, target_names=['Non-Duplicate', 'Duplicate']))

### 5.3 Confusion Matrix

In [ ]:
svm_cm = confusion_matrix(y_test, svm_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    svm_cm,
    annot=True,
    fmt='d',
    cmap='Oranges',
    xticklabels=['Non-Dup', 'Duplicate'],
    yticklabels=['Non-Dup', 'Duplicate']
)
plt.title('Linear SVM — Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
## 6. Classical ML Comparison Table

Full side-by-side comparison of both models across all key metrics.

In [ ]:
from sklearn.metrics import precision_score, recall_score

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Linear SVM'],
    'Accuracy': [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, svm_pred)
    ],
    'F1 (Duplicate)': [
        f1_score(y_test, lr_pred),
        f1_score(y_test, svm_pred)
    ],
    'Precision (Dup)': [
        precision_score(y_test, lr_pred),
        precision_score(y_test, svm_pred)
    ],
    'Recall (Dup)': [
        recall_score(y_test, lr_pred),
        recall_score(y_test, svm_pred)
    ],
    'F1 (Non-Dup)': [
        f1_score(y_test, lr_pred, pos_label=0),
        f1_score(y_test, svm_pred, pos_label=0)
    ],
    'Feature Type': ['TF-IDF (1,2)-grams', 'TF-IDF (1,2)-grams']
})

# Round for readability
comparison.set_index('Model', inplace=True)
comparison = comparison.round(4)
comparison

In [ ]:
# Visual bar chart comparison
metrics = ['Accuracy', 'F1 (Duplicate)', 'Precision (Dup)', 'Recall (Dup)']
comparison[metrics].T.plot(
    kind='bar',
    figsize=(10, 5),
    color=['steelblue', 'coral'],
    edgecolor='white',
    width=0.6
)
plt.title('Model Comparison — LR vs SVM (TF-IDF Features)', fontsize=13)
plt.ylabel('Score')
plt.xticks(rotation=15, ha='right')
plt.ylim(0, 1)
plt.legend(title='Model')
plt.tight_layout()
plt.show()

---
## 7. Approach 2 — Word2Vec Embeddings

**Why Word2Vec?**  
TF-IDF treats each word independently and ignores word meaning.  
**Word2Vec** trains a neural network to learn dense vector representations (embeddings) where  
semantically similar words are close together in vector space.

For example: `king - man + woman ≈ queen`

**Strategy for sentence pairs:**  
Since Word2Vec gives vectors per word, we represent each question as the **mean of its word vectors**.  
Then the feature vector for each pair = `[mean(q1_vectors), mean(q2_vectors), abs_diff, element_wise_product]`

This gives the model signal about both the individual question representations and their similarity.

### 7.1 Tokenize for Word2Vec Training

In [ ]:
# Word2Vec needs a list of token lists
def simple_tokenize(text):
    return text.lower().split()

# Combine all questions from both columns for Word2Vec training
all_sentences = (
    df['clean_q1'].apply(simple_tokenize).tolist() +
    df['clean_q2'].apply(simple_tokenize).tolist()
)

print(f'Total sentences for Word2Vec training: {len(all_sentences):,}')
print(f'Sample tokens: {all_sentences[0][:10]}')

### 7.2 Train Word2Vec Model

In [ ]:
w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=100,   # 100-dimensional word vectors
    window=5,          # context window: 5 words left/right
    min_count=2,       # ignore words appearing fewer than 2 times
    workers=4,         # parallel training
    epochs=5,
    seed=42
)

vocab_size = len(w2v_model.wv)
print(f'Word2Vec training complete.')
print(f'Vocabulary size: {vocab_size:,}')
print(f'Vector dimension: {w2v_model.vector_size}')

### 7.3 Explore Word2Vec Embeddings

In [ ]:
# Most similar words to 'india'
print('Words most similar to "india":')
for word, score in w2v_model.wv.most_similar('india', topn=5):
    print(f'  {word:<20} similarity = {score:.4f}')

print()

# Most similar words to 'best'
print('Words most similar to "best":')
for word, score in w2v_model.wv.most_similar('best', topn=5):
    print(f'  {word:<20} similarity = {score:.4f}')

### 7.4 Build Word2Vec Feature Matrix

Strategy: represent each question as the **mean of its word vectors**.  
Then combine: `[q1_vec, q2_vec, |q1-q2|, q1*q2]` → 400-dimensional feature vector per row.

In [ ]:
def sentence_vector(tokens, model, vector_size):
    """Compute mean word vector for a list of tokens."""
    vecs = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]
    if vecs:
        return np.mean(vecs, axis=0)
    return np.zeros(vector_size)

VECTOR_SIZE = w2v_model.vector_size

print('Building Word2Vec feature matrix...')

q1_vecs = np.vstack([
    sentence_vector(simple_tokenize(text), w2v_model, VECTOR_SIZE)
    for text in df['clean_q1']
])

q2_vecs = np.vstack([
    sentence_vector(simple_tokenize(text), w2v_model, VECTOR_SIZE)
    for text in df['clean_q2']
])

# Combine: absolute difference + element-wise product capture similarity signal
abs_diff = np.abs(q1_vecs - q2_vecs)
elem_prod = q1_vecs * q2_vecs

X_w2v = np.hstack([q1_vecs, q2_vecs, abs_diff, elem_prod])

print(f'Word2Vec feature matrix shape: {X_w2v.shape}')

### 7.5 Train-Test Split (Word2Vec)

In [ ]:
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_w2v, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train size: {X_train_w2v.shape[0]:,}  |  Test size: {X_test_w2v.shape[0]:,}')

### 7.6 Train LR + SVM on Word2Vec Features

In [ ]:
# Logistic Regression on Word2Vec
lr_w2v = LogisticRegression(max_iter=1000)
lr_w2v.fit(X_train_w2v, y_train_w2v)
lr_w2v_pred = lr_w2v.predict(X_test_w2v)

print('=== LR + Word2Vec Results ===')
print(f'Accuracy : {accuracy_score(y_test_w2v, lr_w2v_pred):.4f}')
print(f'F1 Score : {f1_score(y_test_w2v, lr_w2v_pred):.4f}')
print()
print(classification_report(y_test_w2v, lr_w2v_pred, target_names=['Non-Duplicate', 'Duplicate']))

In [ ]:
# Linear SVM on Word2Vec
svm_w2v = LinearSVC(max_iter=1000)
svm_w2v.fit(X_train_w2v, y_train_w2v)
svm_w2v_pred = svm_w2v.predict(X_test_w2v)

print('=== SVM + Word2Vec Results ===')
print(f'Accuracy : {accuracy_score(y_test_w2v, svm_w2v_pred):.4f}')
print(f'F1 Score : {f1_score(y_test_w2v, svm_w2v_pred):.4f}')
print()
print(classification_report(y_test_w2v, svm_w2v_pred, target_names=['Non-Duplicate', 'Duplicate']))

---
## 8. Full Classical ML Comparison Table

All 4 model variants compared: 2 models × 2 feature types.

In [ ]:
full_comparison = pd.DataFrame({
    'Model': [
        'LR + TF-IDF',
        'SVM + TF-IDF',
        'LR + Word2Vec',
        'SVM + Word2Vec'
    ],
    'Feature Type': [
        'TF-IDF (1,2)-grams',
        'TF-IDF (1,2)-grams',
        'Word2Vec (mean)',
        'Word2Vec (mean)'
    ],
    'Accuracy': [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, svm_pred),
        accuracy_score(y_test_w2v, lr_w2v_pred),
        accuracy_score(y_test_w2v, svm_w2v_pred)
    ],
    'F1 (Dup)': [
        f1_score(y_test, lr_pred),
        f1_score(y_test, svm_pred),
        f1_score(y_test_w2v, lr_w2v_pred),
        f1_score(y_test_w2v, svm_w2v_pred)
    ],
    'Precision (Dup)': [
        precision_score(y_test, lr_pred),
        precision_score(y_test, svm_pred),
        precision_score(y_test_w2v, lr_w2v_pred),
        precision_score(y_test_w2v, svm_w2v_pred)
    ],
    'Recall (Dup)': [
        recall_score(y_test, lr_pred),
        recall_score(y_test, svm_pred),
        recall_score(y_test_w2v, lr_w2v_pred),
        recall_score(y_test_w2v, svm_w2v_pred)
    ],
    'F1 (Non-Dup)': [
        f1_score(y_test, lr_pred, pos_label=0),
        f1_score(y_test, svm_pred, pos_label=0),
        f1_score(y_test_w2v, lr_w2v_pred, pos_label=0),
        f1_score(y_test_w2v, svm_w2v_pred, pos_label=0)
    ]
})

full_comparison.set_index('Model', inplace=True)
full_comparison = full_comparison.round(4)
full_comparison

In [ ]:
# Full comparison bar chart
metrics_to_plot = ['Accuracy', 'F1 (Dup)', 'Precision (Dup)', 'Recall (Dup)']
full_comparison[metrics_to_plot].T.plot(
    kind='bar',
    figsize=(12, 5),
    color=['steelblue', 'coral', 'mediumseagreen', 'mediumpurple'],
    edgecolor='white',
    width=0.7
)
plt.title('All Models Comparison — TF-IDF vs Word2Vec', fontsize=13)
plt.ylabel('Score')
plt.xticks(rotation=15, ha='right')
plt.ylim(0, 1)
plt.legend(title='Model', loc='lower right')
plt.tight_layout()
plt.show()

---
## 9. Error Analysis — Where Do Models Fail?

Understanding **error types** is as important as the accuracy score.  

- **False Negative (FN):** Model predicted Non-Duplicate but actual = Duplicate  
  → These are missed duplicates — the hardest cases (paraphrases with different vocabulary)

- **False Positive (FP):** Model predicted Duplicate but actual = Non-Duplicate  
  → These are false alarms — questions that share words but ask different things

In [ ]:
# Build results DataFrame for SVM (best TF-IDF model)
results = df.iloc[y_test.index].copy()
results['Predicted'] = svm_pred
results['Correct'] = (results['label'] == results['Predicted'])

# Separate error types
false_negatives = results[(results['label'] == 1) & (results['Predicted'] == 0)]
false_positives = results[(results['label'] == 0) & (results['Predicted'] == 1)]

print(f'Total errors    : {(~results["Correct"]).sum():,}')
print(f'False Negatives : {len(false_negatives):,}  (missed duplicates)')
print(f'False Positives : {len(false_positives):,}  (false alarms)')

### 9.1 False Negative Examples (Missed Duplicates)

These are question pairs that ARE duplicates but the model predicted non-duplicate.  
Notice how they use **different vocabulary** to express the same idea — TF-IDF cannot detect this.

In [ ]:
print('=== FALSE NEGATIVES — Missed Duplicate Pairs ===')
print('(Model said Non-Duplicate, but they ARE duplicates)\n')

for _, row in false_negatives[['sentence1', 'sentence2']].head(5).iterrows():
    print(f'Q1: {row["sentence1"]}')
    print(f'Q2: {row["sentence2"]}')
    print(f'→ WHY MISSED: different words, same meaning (TF-IDF sees low overlap)')
    print()

### 9.2 False Positive Examples (False Alarms)

These are pairs that are NOT duplicates but the model predicted duplicate.  
Notice how they share similar words but ask about **different things**.

In [ ]:
print('=== FALSE POSITIVES — False Alarm Pairs ===')
print('(Model said Duplicate, but they are NOT duplicates)\n')

for _, row in false_positives[['sentence1', 'sentence2']].head(5).iterrows():
    print(f'Q1: {row["sentence1"]}')
    print(f'Q2: {row["sentence2"]}')
    print(f'→ WHY WRONG: similar vocabulary, different intent (TF-IDF sees high overlap)')
    print()

### 9.3 Error Summary Table

In [ ]:
errors_all = results[results['label'] != results['Predicted']]
errors_all[['clean_q1', 'clean_q2', 'label', 'Predicted']].head(10)

---
## 10. Save Results

In [ ]:
# Save full comparison table
full_comparison.to_parquet('week2_model_comparison.parquet', index=True)
print('Saved: week2_model_comparison.parquet')

# Save error examples
errors_all[['sentence1', 'sentence2', 'label', 'Predicted']].to_parquet(
    'week2_error_examples.parquet', index=False
)
print('Saved: week2_error_examples.parquet')

---
## 11. Summary & Key Findings

### What We Built
| Component | Details |
|---|---|
| Feature Type 1 | TF-IDF with unigrams + bigrams, max 5000 features |
| Feature Type 2 | Word2Vec (100-dim, mean pooling, diff + product) |
| Model 1 | Logistic Regression (max_iter=1000) |
| Model 2 | Linear SVM (max_iter=1000) |
| Split | 80/20 stratified, random_state=42 |

### Key Findings
1. **TF-IDF models outperform Word2Vec** on this dataset — word overlap is highly informative for Quora questions
2. **SVM slightly outperforms LR** on both feature types — maximum-margin objective is well-suited for text classification
3. **False Negatives dominate errors** — the model misses duplicates that use different vocabulary (paraphrases)
4. **Word2Vec captures some semantic signal** but mean pooling loses sentence structure — this is a key limitation
5. **Core limitation:** Both TF-IDF and Word2Vec fail to detect semantic paraphrases where questions share meaning but not words

### Next Steps (Week 3)
- Sentence-BERT / sentence embeddings (captures full semantic meaning)
- Cross-encoder models specifically designed for sentence-pair classification
- Siamese neural networks for pairwise similarity